In [1]:
import datetime

import pandas as pd
import requests

# Upload news from tradingview

In [2]:
def get_news_for_a_period(start_date: str, end_date: str, countries=None) -> pd.DataFrame:
    if countries is None:
        countries = ['US', 'FR', 'GB', 'EU', 'AU', 'DE', 'JP', 'CN']
        
    start_date += 'T00:00:00'
    end_date += 'T00:00:00'

    url = 'https://economic-calendar.tradingview.com/events'
    headers = {
        'Origin': 'https://in.tradingview.com'
    }
    payload = {
        'from': start_date + '.000Z',
        'to': end_date + '.000Z',
        'countries': ','.join(countries)
    }
    data = requests.get(url, headers=headers, params=payload).json()
    data = pd.DataFrame(data['result'])
    return data

In [3]:
def get_news_for_today(countries=None):
    today = datetime.date.today()
    tomorrow = today + datetime.timedelta(days=1)
    
    today = today.strftime('%Y-%m-%d')
    tomorrow = tomorrow.strftime('%Y-%m-%d')
    
    news = get_news_for_a_period(today, tomorrow, countries)
    return news

In [4]:
today_news = get_news_for_today()

In [5]:
today_news

,id,title,country,indicator,comment,category,period,referenceDate,source,source_url,actual,previous,forecast,actualRaw,previousRaw,forecastRaw,currency,importance,date,ticker
0,401347,S&P Global Manufacturing PMI Final,AU,manufacturing pmi,The S&P Global Australia Manufacturing PMI is ...,bsnss,Jan,2026-01-31T00:00:00Z,S&P Global,https://www.pmi.spglobal.com/public,None,51.6,None,None,51.6,None,AUD,-1,2026-02-01T22:00:00.000Z,NaN
1,393949,BoJ Summary of Opinions,JP,Interest Rate,"In Japan, interest rates are set by the Bank o...",mny,,None,Central Bank,https://www.boj.or.jp,None,NaN,None,None,NaN,None,JPY,0,2026-02-01T23:50:00.000Z,ECONOMICS:JPINTR


# Upload prices from Twelve Data

In [6]:
from twelvedata import TDClient

In [7]:
api = '8555d5ea4f114fadab8acf39cf0eac65'

In [9]:
# Initialize client - apikey parameter is requiered
td = TDClient(apikey=api)

In [48]:
ts = td.time_series(
    symbol=['EUR/USD', 'GBP/USD', 'USD/CHF', 'USD/JPY', 'USD/CAD', 'AUD/USD', 'NZD/USD'],
    interval="30min",
    outputsize=15
)

In [65]:
df = ts.as_pandas()
df.reset_index(inplace=True)
df.columns = ['symbol', 'datetime', 'open', 'high', 'low', 'close']
df.sort_values(by=['symbol', 'datetime'], inplace=True)

In [66]:
df

,symbol,datetime,open,high,low,close
89,AUD/USD,2026-02-01 02:30:00,0.69608,0.69611,0.69606,0.69610
88,AUD/USD,2026-02-01 03:00:00,0.69608,0.69611,0.69605,0.69606
87,AUD/USD,2026-02-01 03:30:00,0.69606,0.69611,0.69605,0.69610
86,AUD/USD,2026-02-01 04:00:00,0.69608,0.69775,0.69607,0.69677
85,AUD/USD,2026-02-01 04:30:00,0.69678,0.69681,0.69644,0.69644
...,...,...,...,...,...,...
49,USD/JPY,2026-02-01 07:30:00,154.68821,154.70450,154.68611,154.70005
48,USD/JPY,2026-02-01 08:00:00,154.69957,154.70540,154.62167,154.67739
47,USD/JPY,2026-02-01 08:30:00,154.67597,154.71985,154.67390,154.69889
46,USD/JPY,2026-02-01 09:00:00,154.68960,154.70023,154.62035,154.69451


In [51]:
import talib as ta

In [52]:
def calculate_atr_for_symbols(df, symbol_column='symbol', high_column='high', low_column='low', close_column='close', timeperiod=14):
    """
    Calculate ATR for each symbol in the DataFrame and return the last row with prices and ATR value for each symbol.

    Args:
        df (pd.DataFrame): Input DataFrame with columns for symbol, high, low, and close prices.
        symbol_column (str): Column name for the symbol.
        high_column (str): Column name for the high prices.
        low_column (str): Column name for the low prices.
        close_column (str): Column name for the close prices.
        timeperiod (int): Time period for ATR calculation.

    Returns:
        pd.DataFrame: DataFrame with the last row of prices and ATR value for each symbol.
    """
    result = []

    # Group by symbol
    for symbol, group in df.groupby(symbol_column):
        # Ensure the group is sorted by time (if applicable)
        group = group.sort_index()

        # Calculate ATR
        atr = ta.ATR(group[high_column], group[low_column], group[close_column], timeperiod=timeperiod)

        # Add ATR to the group
        group = group.assign(ATR=atr)

        # Append the last row with ATR value
        result.append(group.iloc[-1])

    # Combine results into a single DataFrame
    return pd.DataFrame(result)

In [53]:
calculate_atr_for_symbols(df)

,symbol,datetime,open,high,low,close,ATR
89,AUD/USD,2026-02-01 02:30:00,0.69608,0.69611,0.69606,0.69610,0.000818
14,EUR/USD,2026-02-01 02:30:00,1.18551,1.18556,1.18517,1.18535,0.000153
29,GBP/USD,2026-02-01 02:30:00,1.36910,1.36915,1.36868,1.36890,0.000576
104,NZD/USD,2026-02-01 02:30:00,0.60215,0.60217,0.60213,0.60214,0.000492
74,USD/CAD,2026-02-01 02:30:00,1.36128,1.36167,1.36126,1.36157,0.001985
44,USD/CHF,2026-02-01 02:30:00,0.77241,0.77260,0.77240,0.77260,0.000661
59,USD/JPY,2026-02-01 02:30:00,154.73591,154.76595,154.73385,154.75006,0.083506


In [67]:
df['atr'] = df.groupby('symbol').apply(lambda x: ta.ATR(x['high'], x['low'], x['close'], timeperiod=14), include_groups=False).reset_index(drop=True)

In [68]:
df.dropna(subset=['atr'], inplace=True)

In [69]:
df

,symbol,datetime,open,high,low,close,atr
89,AUD/USD,2026-02-01 02:30:00,0.69608,0.69611,0.69606,0.69610,0.000539
14,EUR/USD,2026-02-01 02:30:00,1.18551,1.18556,1.18517,1.18535,0.000762
29,GBP/USD,2026-02-01 02:30:00,1.36910,1.36915,1.36868,1.36890,0.000114
104,NZD/USD,2026-02-01 02:30:00,0.60215,0.60217,0.60213,0.60214,0.075247
74,USD/CAD,2026-02-01 02:30:00,1.36128,1.36167,1.36126,1.36157,0.001771
44,USD/CHF,2026-02-01 02:30:00,0.77241,0.77260,0.77240,0.77260,0.000507
59,USD/JPY,2026-02-01 02:30:00,154.73591,154.76595,154.73385,154.75006,0.000421
